In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from tqdm import tqdm
import pandas as pd
import time
import numpy as np

##Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Synthcity
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")

#Fit GAN using ALL training data
from synthcity.plugins import Plugins

syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################

def dummify_columns(df):
    # Dummify marital and educational
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    # Drop transformed cols
    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):
    # Split into features and target for train and test datasets
    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']

        
    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    
    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store metrics
    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-04 19:06:23,307 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmpdjtd7y4t
2023-08-04 19:06:23,308 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmpdjtd7y4t/_remote_module_non_scriptable.py


In [2]:
# THE other FASST extrinisc15 dataset had random state = 4

countlen = len(df)*0.7*0.925

syn_sizes = [1, 0.5*countlen, 1*countlen,  3*countlen,  5*countlen, 
            8*countlen, 12*countlen, 18*countlen, 32*countlen, 48*countlen, 64*countlen]

n_iterations = 100 # Number of bootstrapping iterations

# Placeholder for the results
results = []
syn_model = Plugins().get('ctgan')
# Bootstrap iteration loop
for i in range(n_iterations):
    # Resample entire dataset
    start_time = time.time()
    
    # Perform train/test split
    df_train_main, df_test = train_test_split(df, test_size=0.3, random_state=i*124)
    df_train_1, df_train_2 = train_test_split(df_train_main, test_size=0.075, random_state=i*1244)

    # Resample
    #df_test_resample = resample(df_test,replace=True)
    #df_train_1_resample = resample(df_train_1,replace=True)
    #df_train_2_resample = resample(df_train_2,replace=True)
    
    loader = GenericDataLoader(df_train_1, target_column='Response')
   
    syn_model.fit(loader)
    # Loop through the different synthetic set sizes
    for size in syn_sizes:
        # Generate synthetic set of the current size based on resampled training data
        #syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()
        syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()

        print(syn_set['Response'].mean() * 100)

        # Add synthetic data to resampled training data
        df_train_combined = pd.concat([df_train_2, syn_set], axis=0)

        # Dummify train and test datasets
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_exc_df_1 = pd.DataFrame(results)

 37%|██████████████▉                         | 749/2000 [05:35<09:20,  2.23it/s]


0.0
9.1164095371669
10.2312543798178
10.063039925286015
9.80666853460353
9.894054811312495
10.168106467429372
9.973150706253161
10.002845448377
10.220192321723017
10.252257181942545
Time: 574.963812828064 seconds
Iteration: 0 


 30%|███████████▉                            | 599/2000 [04:42<11:00,  2.12it/s]


100.0
14.446002805049089
14.716187806587246
14.429138454354423
14.738021854861305
14.552140793275544
15.01867849638104
14.806023580684075
14.680324818876267
14.297179378675345
14.584952120383038
Time: 533.2524170875549 seconds
Iteration: 1 


 42%|████████████████▉                       | 849/2000 [06:00<08:09,  2.35it/s]


...

0.0
18.653576437587656
18.149964961457606
18.234882091991594
18.352479686186605
18.001926276157953
17.86131216437077
17.85283474065139
17.449164970341673
17.745253972654712
17.60218878248974
Time: 831.5066080093384 seconds
Iteration: 97 


 40%|███████████████▉                        | 799/2000 [06:18<09:29,  2.11it/s]


0.0
15.287517531556801
15.837421163279608
16.18024749007705
15.94284113196974
15.567813676560721
15.468129815549847
15.94614576442663
15.726574299034738
15.636719148998262
15.835841313269494
Time: 675.0131766796112 seconds
Iteration: 98 


 32%|████████████▉                           | 649/2000 [05:40<11:49,  1.90it/s]


0.0
11.079943899018232
12.263489838822705
11.627363997198225
11.30568786775007
12.03922598721653
11.236282979220173
11.654149966924782
11.749512990566243
11.673549196713896
11.567715458276334
Time: 631.214277267456 seconds
Iteration: 99 


In [3]:
results_exc_df_1.to_clipboard()

In [ ]:
np.shape(df_train_combined)